# 🧠 Demo 05 — Agentic RAG & Décomposition de Requêtes

Démonstration du RAG agentique avec décomposition intelligente des requêtes complexes.

**Concepts démontrés :**
- Décomposition automatique de questions complexes
- Exécution de sous-requêtes
- Agrégation et déduplication des résultats
- Comparaison RAG simple vs RAG agentique

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path('.').resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from src.agents.query_decomposer import FinancialQueryDecomposer
from src.retrieval.vector_store import FinancialVectorStore
from src.retrieval.retriever import HybridFinancialRetriever
from src.retrieval.reranker import CrossEncoderReRanker
from src.generation.generator import FinancialAnswerGenerator
from src.agents.rag_agent import FinancialRAGAgent

vs = FinancialVectorStore()
retriever = HybridFinancialRetriever(vector_store=vs)
reranker = CrossEncoderReRanker()
agent = FinancialRAGAgent(vs, retriever, reranker)
decomposer = FinancialQueryDecomposer()

print('Agent RAG initialisé')

## 1. Décomposition de Requêtes

In [ ]:
complex_questions = [
    "Quel est le CA d'Apple en FY2023 ?",  # Simple
    "Comparez la croissance du CA d'Apple et Microsoft sur leurs derniers exercices.",  # Complex
    "Quel est l'impact de l'IA sur les revenus de Microsoft en 2024 ?",  # Moderate
]

for q in complex_questions:
    needs = decomposer.needs_decomposition(q)
    print(f'Q: {q[:60]}...')
    print(f'   Nécessite décomposition: {needs}')
    if needs:
        sqs = decomposer.decompose(q)
        print(f'   Sous-requêtes ({len(sqs)}):')
        for sq in sqs:
            print(f'     [{sq.type}] {sq.query} | entities={sq.entities}')
    print()

## 2. Comparaison RAG Simple vs Agentique

In [ ]:
import time
from IPython.display import Markdown, display

complex_question = "Comparez la croissance d'Apple et Microsoft sur leurs résultats financiers récents."

# RAG Simple (sans décomposition)
start = time.time()
answer_simple = agent.answer(complex_question, use_decomposition=False)
time_simple = time.time() - start

# RAG Agentique (avec décomposition)
start = time.time()
answer_agentic = agent.answer(complex_question, use_decomposition=True)
time_agentic = time.time() - start

print('=== RAG Simple ===')
print(f'Temps: {time_simple:.2f}s | Confiance: {answer_simple.confidence_score:.0%} | Sources: {answer_simple.context_docs_count}')

print('\n=== RAG Agentique ===')
print(f'Temps: {time_agentic:.2f}s | Confiance: {answer_agentic.confidence_score:.0%} | Sources: {answer_agentic.context_docs_count}')
if answer_agentic.sub_queries:
    print(f'Sous-requêtes utilisées: {len(answer_agentic.sub_queries)}')
    for sq in answer_agentic.sub_queries:
        print(f'  → {sq}')

## 3. Réponse Agentique Complète

In [ ]:
display(Markdown('### Réponse RAG Agentique\n\n' + answer_agentic.answer))

## 4. Visualisation du Flux Agentique

In [ ]:
import plotly.graph_objects as go

# Compare simple vs agentic metrics
categories = ['Confiance', 'Nb Sources', 'Sous-requêtes']

values_simple = [
    answer_simple.confidence_score * 100,
    answer_simple.context_docs_count * 10,
    0,
]
values_agentic = [
    answer_agentic.confidence_score * 100,
    answer_agentic.context_docs_count * 10,
    len(answer_agentic.sub_queries) * 20,
]

fig = go.Figure()
fig.add_trace(go.Bar(
    name='RAG Simple',
    x=categories,
    y=values_simple,
    marker_color='#3B82F6',
))
fig.add_trace(go.Bar(
    name='RAG Agentique',
    x=categories,
    y=values_agentic,
    marker_color='#F0B429',
))

fig.update_layout(
    title='Comparaison RAG Simple vs Agentique',
    template='plotly_dark',
    barmode='group',
    yaxis_title='Score (normalisé)',
)
fig.show()

## 5. Sources Pertinentes Identifiées

In [ ]:
sources = agent.get_relevant_sources(complex_question)
if sources:
    import pandas as pd
    df_sources = pd.DataFrame(sources)
    print('Sources pertinentes identifiées:')
    print(df_sources[['filename', 'score', 'date', 'doc_type', 'excerpt']].to_string(index=False))
else:
    print('Aucune source trouvée. Vérifiez que des documents sont indexés.')